# Notebook 2: GEDI Forest Structure MODIS Aggregation

Loads the high-fidelity 25m native-scale gridded GEDI stack assets from Stage 1 (`GediStack_{basin}_{gi}`),
mosaics them seamlessly, applies quality masking directly at 25m (JRC TMF undisturbed forest +
slope <10° terrain filter), aggregates the masked structural indicators to MODIS resolution
(~463m equivalent), and exports the final 4-band GEDI MODIS asset per basin.

### Design Principles:
1. **Option C High-Precision Masking**: All masks applied at 25m native resolution before reduction to prevent spatial leakage.
2. **Slope Quality Filter**: GEDI waveform metrics degrade on steep terrain (slope ≥10°) due to footprint-scale elevation spread biasing RH98 and PAVD estimates.
3. **Functional Encapsulation (Block 2)**: All processing steps are wrapped in pure, testable functions.
4. **Unit Tested (Block 3)**: Rigorous assertions verify that spatial reduction, bands, and projections align correctly.

### Masks Applied (at 25m before aggregation):
- **JRC TMF Class 10**: Continuously undisturbed moist forest since ~1982
- **SRTM slope < 10°**: Excludes steep terrain where GEDI reliability degrades

### Bands in Output Multi-band Image:
- **`uoi`** (Understory Openness Index): reduced via mean
- **`rh98`** (98th percentile canopy height): reduced via mean
- **`gedi_n`** (Footprint shot count sample size): reduced via sum
- **`uoi_sd`** (Within-pixel UOI standard deviation): reduced via stdDev

In [ ]:
# =============================================================================
# BLOCK 1: SETUP AND CONFIGURATION
# =============================================================================
import ee

try:
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("✓ GEE initialized successfully!")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("✓ GEE initialized successfully!")

# Asset paths
ASSET_ROOT = 'projects/quantum-bonus-434714-t2/assets/DefaunationFromSpace'
JRC_TMF_ASSET = 'projects/JRC/TMF/v1_2024/TransitionMap_MainClasses'

# Study regions
CONGO_BBOX = ee.Geometry.Rectangle([-5, -15, 45, 15])
AMAZON_BBOX = ee.Geometry.Rectangle([-85, -15, -35, 15])
SEA_BBOX = ee.Geometry.Rectangle([90, -15, 140, 15])
BASINS = [('Congo', CONGO_BBOX), ('Amazon', AMAZON_BBOX), ('SE_Asia', SEA_BBOX)]

# Masking constants
FOREST_CLASS_UNDISTURBED = 10  # Continuously undisturbed since ~1982
SLOPE_MAX = 10  # Degrees: GEDI waveform metrics degrade on steep terrain

# Spatial parameters (3x3 grid matching Stage 1 GediStack partitions)
GEDI_GRID_COLS = 3
GEDI_GRID_ROWS = 3
N_GEDI_TILES = GEDI_GRID_COLS * GEDI_GRID_ROWS

# Reference MODIS projection (standard sinusoidal -> EPSG:4326 ~463m grid)
_modis_col = ee.ImageCollection('MODIS/061/MOD17A3HGF').select('Npp')
MODIS_PROJ = ee.Image(_modis_col.first()).projection()
MODIS_SCALE = 463.3127165279165  # MODIS equatorial pixel size in meters

print("✓ Configuration and reference parameters loaded.")


In [ ]:
# =============================================================================
# BLOCK 2: IN-MEMORY COMPUTATION LOGIC
# =============================================================================

def load_and_mosaic_gedi(basin_name):
    """Loads the 9 gridded GediStack tiles and mosaics them into a single 25m image."""
    grids = [
        ee.Image(f'{ASSET_ROOT}/GediStack_{basin_name}_{i}') for i in range(N_GEDI_TILES)
    ]
    return ee.ImageCollection(grids).mosaic()

def apply_quality_masks(gedi_mosaic, basin_name):
    """Applies all quality masks at 25m native resolution before MODIS aggregation.
    
    Three masks are applied:
    1. GEDI shot quality — from pre-exported GediQuality asset (l2b_quality_flag == 1
       AND degrade_flag == 0 for every contributing shot)
    2. JRC TMF Class 10 (undisturbed moist forest) — restricts to intact forest pixels
    3. SRTM slope < 10° — excludes steep terrain where GEDI waveform processing
       degrades due to footprint-scale elevation spread biasing RH98 and PAVD
    
    Both JRC TMF (30m) and SRTM (30m) are dynamically resampled to the GEDI grid (25m) by GEE.
    
    Note: Elevation and HAND filters are analytical scope decisions applied later
    in the R analysis pipeline, not here, to avoid permanently removing pixels.
    """
    # GEDI shot-level quality mask (exported from NB1 Block 5)
    qmask = ee.Image(f'{ASSET_ROOT}/GediQuality_{basin_name}')
    gedi_quality_mask = (qmask.select('quality_min').eq(1)
                         .And(qmask.select('degrade_max').eq(0)))
    
    # Forest intactness mask
    tmf = ee.ImageCollection(JRC_TMF_ASSET).mosaic()
    forest_mask = tmf.eq(FOREST_CLASS_UNDISTURBED)
    
    # Slope quality mask
    srtm = ee.Image('USGS/SRTMGL1_003')
    slope = ee.Terrain.slope(srtm)
    slope_mask = slope.lt(SLOPE_MAX)
    
    # Apply combined mask
    combined_mask = gedi_quality_mask.And(forest_mask).And(slope_mask)
    return gedi_mosaic.updateMask(combined_mask)

def reduce_to_modis_scale(gedi_masked):
    """Aggregates 25m GEDI bands to MODIS scale (~463.3m) using proper spatial reducers.
    
    - UOI and rh98 are averaged (ee.Reducer.mean())
    - Shot counts (N) are summed (ee.Reducer.sum())
    - UOI within-pixel variability via stdDev (ee.Reducer.stdDev())
    """
    # 1. Spatial aggregation for Mean-based indicators: UOI and rh98
    uoi = gedi_masked.select('GEDI_UOI').setDefaultProjection(crs='EPSG:4326', scale=25).reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=65535
    ).reproject(crs=MODIS_PROJ).rename('uoi')
    
    rh98 = gedi_masked.select('GEDI_rh98').setDefaultProjection(crs='EPSG:4326', scale=25).reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=65535
    ).reproject(crs=MODIS_PROJ).rename('rh98')
    
    # 2. Spatial aggregation for sample size footprint shot count (Sum)
    gedi_n = gedi_masked.select('GEDI_N').setDefaultProjection(crs='EPSG:4326', scale=25).reduceResolution(
        reducer=ee.Reducer.sum(), maxPixels=65535
    ).reproject(crs=MODIS_PROJ).rename('gedi_n')
    
    # 3. Within-pixel UOI variability: stdDev of 25m UOI within each MODIS pixel
    uoi_sd = gedi_masked.select('GEDI_UOI').setDefaultProjection(crs='EPSG:4326', scale=25).reduceResolution(
        reducer=ee.Reducer.stdDev(), maxPixels=65535
    ).reproject(crs=MODIS_PROJ).rename('uoi_sd')
    
    # 4. Concatenate into a unified 4-band MODIS image stack
    return ee.Image.cat([uoi, rh98, gedi_n, uoi_sd]).toFloat()

def build_gedi_modis_stack(basin_name):
    """Orchestrates the entire in-memory processing pipeline for a study basin."""
    # Step 1: Load grids and mosaic
    gedi_mosaic = load_and_mosaic_gedi(basin_name)
    
    # Step 2: Apply quality masks at 25m (GEDI shot quality + forest intactness + slope)
    gedi_masked = apply_quality_masks(gedi_mosaic, basin_name)
    
    # Step 3: Spatial reduction to MODIS scale
    return reduce_to_modis_scale(gedi_masked)

print("✓ Functional processing logic loaded.")


In [ ]:
# =============================================================================
# BLOCK 3: UNIT TESTS
# =============================================================================

def run_unit_tests():
    print("Running unit tests (using Congo basin)...\n")
    passed = 0
    failed = 0
    
    # --- Test 1: load_and_mosaic_gedi ---
    try:
        print("  [1/5] Validating load_and_mosaic_gedi()...")
        gedi_mosaic = load_and_mosaic_gedi('Congo')
        assert isinstance(gedi_mosaic, ee.Image), "Mosaic must return an ee.Image"
        mosaic_bands = gedi_mosaic.bandNames().getInfo()
        assert 'GEDI_UOI' in mosaic_bands, "Missing GEDI_UOI band"
        assert 'GEDI_N' in mosaic_bands, "Missing GEDI_N band"
        assert 'GEDI_rh98' in mosaic_bands, "Missing GEDI_rh98 band"
        passed += 1
        print(f"    ✓ Loaded seamless 25m GediStack. Bands: {mosaic_bands}")
    except Exception as e:
        failed += 1
        print(f"    ✗ load_and_mosaic_gedi FAILED: {e}")
        
    # --- Test 2: apply_quality_masks ---
    try:
        print("  [2/5] Validating apply_quality_masks()...")
        gedi_mosaic = load_and_mosaic_gedi('Congo')
        gedi_masked = apply_quality_masks(gedi_mosaic, 'Congo')
        assert isinstance(gedi_masked, ee.Image), "Masked output must return an ee.Image"
        masked_bands = gedi_masked.bandNames().getInfo()
        assert set(masked_bands) == {'GEDI_UOI', 'GEDI_rh98', 'GEDI_N'}, f"Unexpected bands: {masked_bands}"
        passed += 1
        print(f"    ✓ Quality masks applied (GEDI shot quality + JRC TMF undisturbed + slope<{SLOPE_MAX}°)")
    except Exception as e:
        failed += 1
        print(f"    ✗ apply_quality_masks FAILED: {e}")
        
    # --- Test 3: reduce_to_modis_scale ---
    try:
        print("  [3/5] Validating reduce_to_modis_scale()...")
        gedi_mosaic = load_and_mosaic_gedi('Congo')
        gedi_masked = apply_quality_masks(gedi_mosaic, 'Congo')
        stack = reduce_to_modis_scale(gedi_masked)
        assert isinstance(stack, ee.Image), "MODIS stack must return an ee.Image"
        stack_bands = stack.bandNames().getInfo()
        assert len(stack_bands) == 4, f"Expected 4 bands, got {len(stack_bands)}"
        assert stack_bands == ['uoi', 'rh98', 'gedi_n', 'uoi_sd'], f"Unexpected bands: {stack_bands}"
        passed += 1
        print(f"    ✓ Aggregated 4-band MODIS stack built. Bands: {stack_bands}")
    except Exception as e:
        failed += 1
        print(f"    ✗ reduce_to_modis_scale FAILED: {e}")
        
    # --- Test 4: build_gedi_modis_stack orchestration ---
    try:
        print("  [4/5] Validating full pipeline orchestration build_gedi_modis_stack()...")
        stack = build_gedi_modis_stack('Congo')
        assert isinstance(stack, ee.Image), "Orchestrated stack must return an ee.Image"
        stack_bands = stack.bandNames().getInfo()
        assert stack_bands == ['uoi', 'rh98', 'gedi_n', 'uoi_sd'], f"Unexpected orchestrated bands: {stack_bands}"
        passed += 1
        print("    ✓ Full orchestration pipeline verified (forest + slope masks + MODIS reduction)")
    except Exception as e:
        failed += 1
        print(f"    ✗ build_gedi_modis_stack FAILED: {e}")
        

    # --- Test 5: Quality mask polarity (keep good, remove bad) ---
    try:
        print("  [5/5] Validating quality mask polarity (synthetic pixel test)...")
        
        # Create a tiny synthetic 2-pixel image: one good, one bad
        # Pixel 1: quality_min=1, degrade_max=0 -> KEEP
        # Pixel 2: quality_min=0, degrade_max=1 -> REMOVE
        good_q = ee.Image.constant(1).rename('quality_min').toUint8()
        good_d = ee.Image.constant(0).rename('degrade_max').toUint8()
        bad_q  = ee.Image.constant(0).rename('quality_min').toUint8()
        bad_d  = ee.Image.constant(1).rename('degrade_max').toUint8()
        
        # Test the mask logic directly (mirrors apply_quality_masks)
        good_mask = good_q.eq(1).And(good_d.eq(0))
        bad_mask  = bad_q.eq(1).And(bad_d.eq(0))
        
        test_pt = ee.Geometry.Point([20.0, 0.0])
        
        good_val = good_mask.reduceRegion(
            reducer=ee.Reducer.first(), geometry=test_pt, scale=25
        ).getInfo()['quality_min']
        
        bad_val = bad_mask.reduceRegion(
            reducer=ee.Reducer.first(), geometry=test_pt, scale=25
        ).getInfo()['quality_min']
        
        assert good_val == 1, f"Good pixel should be 1 (keep), got {good_val}"
        assert bad_val == 0, f"Bad pixel should be 0 (mask), got {bad_val}"
        
        # Also test: degrade_max > 0 alone should mask
        degrade_only_mask = good_q.eq(1).And(ee.Image.constant(5).rename('degrade_max').toUint8().eq(0))
        degrade_val = degrade_only_mask.reduceRegion(
            reducer=ee.Reducer.first(), geometry=test_pt, scale=25
        ).getInfo()['quality_min']
        assert degrade_val == 0, f"Degraded pixel should be 0 (mask), got {degrade_val}"
        
        passed += 1
        print("    \u2713 Polarity verified: quality=1/degrade=0 -> KEEP, quality=0 or degrade>0 -> MASK")
    except Exception as e:
        failed += 1
        print(f"    \u2717 Quality mask polarity FAILED: {e}")
    total = passed + failed
    print(f"\n{'='*60}")
    if failed == 0:
        print(f"  ✓ ALL {passed}/{total} TESTS PASSED SUCCESSFULLY!")
        print("  Ready to export GEDI MODIS stacks.")
    else:
        print(f"  ✗ {passed}/{total} passed, {failed} failed. Fix failures or ensure Stage 1 GediStack exports are done.")
    print(f"{'='*60}")

run_unit_tests()


In [ ]:
# =============================================================================
# BLOCK 4: EXPORT TO ASSETS
# =============================================================================

def safe_start(task, asset_id):
    """Deletes existing asset (if any) then starts export task.
    Prevents 'Asset already exists' failures on re-runs."""
    try:
        ee.data.deleteAsset(asset_id)
        print(f"    Deleted existing: {asset_id.split('/')[-1]}")
    except Exception:
        pass
    task.start()

def export_gedi_modis_stacks(dry_run=True):
    """Launches exports for GEDI MODIS-scale stacks directly to GEE Assets.
    
    Exports 1 multi-band MODIS-scale image per basin (Congo and Amazon).
    Uses safe_start() for asset exports to handle re-runs gracefully.
    """
    tasks = []
    
    for basin_name, basin_geom in BASINS:
        # 1. Compile the 4-band MODIS stack
        stack = build_gedi_modis_stack(basin_name)
        
        # 2. Configure GEE Asset export (static MODIS GEDI stack)
        asset_id = f'{ASSET_ROOT}/GediUndisturbedModis_{basin_name}'
        asset_task = ee.batch.Export.image.toAsset(
            image=stack,
            description=f'gedi_undisturbed_modis_asset_{basin_name}',
            assetId=asset_id,
            region=basin_geom,
            scale=MODIS_SCALE,
            crs='EPSG:4326',
            maxPixels=1e10
        )
        tasks.append((asset_task, f'Asset: GediUndisturbedModis_{basin_name}', asset_id))
        
    print(f'✓ {len(tasks)} MODIS GEDI export tasks configured (Assets):')
    for _, desc, _ in tasks:
        print(f'  - {desc}')
        
    if dry_run:
        print('\nDRY RUN. Call export_gedi_modis_stacks(dry_run=False) to launch tasks.')
    else:
        for task, desc, asset_id in tasks:
            safe_start(task, asset_id)
            print(f'  ✓ Started export: {desc}')
        print('\n✓ All MODIS GEDI exports started!')
        print('  Monitor at: https://code.earthengine.google.com/tasks')

export_gedi_modis_stacks(dry_run=True)
